In [0]:
from pyspark.sql.functions import col, lit, concat, lower, regexp_replace, split, when, current_timestamp, row_number, coalesce, to_timestamp
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType

print("🧪 Starting Bronze Layer Unit Tests (Current Production Logic)...")

# ---------------------------------------------------------
# 1. Setup Mock Target Table (Initial State)
# ---------------------------------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS test_bronze")
spark.sql("DROP TABLE IF EXISTS test_bronze.crm_accounts")

# Base Tracking Schema (Notice: Missing 'a_risk_level__c' to test schema evolution)
spark.sql("""
    CREATE TABLE test_bronze.crm_accounts (
        tenant BIGINT,
        id STRING,
        source_system STRING,
        source_system_object STRING,
        source_key_name STRING,
        source_key_value STRING,
        data_timestamp TIMESTAMP,
        systemmodstamp TIMESTAMP,
        created_date TIMESTAMP,
        created_at TIMESTAMP,
        updated_at TIMESTAMP,
        status STRING,
        domain STRING,
        name STRING,
        a_industry STRING
    ) USING DELTA
""")

# Insert EXISTING123 (To test Partial Updates via MERGE COALESCE)
# Insert STALE_TEST (To test the Sad Path MERGE Timestamp Guard)
spark.sql("""
    INSERT INTO test_bronze.crm_accounts VALUES 
    (1, 'salesforce_Account_Id_EXISTING123', 'salesforce', 'Account', 'Id', 'EXISTING123', CAST('2026-03-01T10:00:00.000Z' AS TIMESTAMP), CAST('2026-03-01T10:00:00.000Z' AS TIMESTAMP), current_timestamp(), current_timestamp(), current_timestamp(), 'new', 'old-domain.com', 'Old Name', 'Retail'),
    (1, 'salesforce_Account_Id_STALE_TEST', 'salesforce', 'Account', 'Id', 'STALE_TEST', CAST('2026-03-05T10:00:00.000Z' AS TIMESTAMP), CAST('2026-03-05T10:00:00.000Z' AS TIMESTAMP), current_timestamp(), current_timestamp(), current_timestamp(), 'new', 'stale.com', 'Current Name', 'Technology')
""")

print("✅ Mock Target Table Created with 2 Initial Records.")

# ---------------------------------------------------------
# 2. Create Mock Incoming Batch (Landing Zone Payload)
# ---------------------------------------------------------
incoming_data = [
    # 🟢 TC1: Intra-batch dedup (Newest extraction time should win)
    ("DUP456", "Duplicate Co", "https://dup.com", "Finance", None, "2026-03-02T10:00:00.000Z", "2026-03-02T10:05:00.000Z"),
    ("DUP456", "Duplicate Co Updated", "https://dup.com", "Finance", None, "2026-03-02T10:00:00.000Z", "2026-03-02T12:05:00.000Z"),
    
    # 🟢 TC2: Partial Update via MERGE
    # Newer timestamp, but Name is NULL. COALESCE should keep 'Old Name' but update Industry to 'Tech'.
    ("EXISTING123", None, "https://new-domain.com", "Tech", None, "2026-03-05T10:00:00.000Z", "2026-03-05T10:05:00.000Z"),
    
    # 🔴 TC3: Stale Data (MERGE Guard)
    # Incoming timestamp is older than the target table (Feb 1 vs March 5). Should be IGNORED.
    ("STALE_TEST", "Stale Name Update", "https://stale-update.com", "Technology", None, "2026-02-01T10:00:00.000Z", "2026-02-01T10:05:00.000Z"),
    
    # 🟢 TC4: Brand New Record + Schema Evolution + Domain Regex
    ("NEW789", "Brand New Co", "https://www.google.com/search?q=test", "Healthcare", "High Risk", "2026-03-10T10:00:00.000Z", "2026-03-10T10:05:00.000Z"),
    
    # 🔴 TC5: Malformed Website URL
    # Regex fails. Domain extraction should fallback to original string.
    ("BAD_URL", "Bad URL Co", "not-a-valid-url", "Unknown", None, "2026-03-11T10:00:00.000Z", "2026-03-11T10:05:00.000Z")
]

schema = StructType([
    StructField("Id", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Website", StringType(), True),
    StructField("Industry", StringType(), True),
    StructField("Risk_Level__c", StringType(), True), 
    StructField("SystemModstamp", StringType(), True),
    StructField("MockFileTime", StringType(), True) 
])

batch_df = spark.createDataFrame(incoming_data, schema)

# ---------------------------------------------------------
# 3. Apply Production Normalization & Dedup Logic
# ---------------------------------------------------------
print("⚙️ Applying Normalization, Schema Evolution, and Deduplication...")

raw_cols = batch_df.columns
df_norm = batch_df.withColumn("status", lit("processing"))

# A. Normalization & Prefixing
df_norm = df_norm \
    .withColumn("tenant", lit(1).cast(LongType())) \
    .withColumn("source_system", lit("salesforce")) \
    .withColumn("source_system_object", lit("Account")) \
    .withColumn("source_key_name", lit("Id")) \
    .withColumn("source_key_value", col("Id")) \
    .withColumn("id", concat(lit("salesforce_Account_Id_"), col("Id"))) \
    .withColumn("data_timestamp", to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX")) \
    .withColumn("systemmodstamp", to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX")) \
    .withColumn("extraction_timestamp", to_timestamp(col("MockFileTime"), "yyyy-MM-dd'T'HH:mm:ss.SSSX")) \
    .withColumn("created_date", current_timestamp()) \
    .withColumn("created_at", current_timestamp()) \
    .withColumn("updated_at", current_timestamp())

# Domain Extraction Regex
clean_url = regexp_replace(regexp_replace(regexp_replace(col("Website"), r'^https?://', ''), r'^www\.', ''), r'/.*$', '')
domain_regex = r'^[a-z0-9][a-z0-9-]*(\.[a-z0-9-]+)*\.[a-z]{2,}$'

df_norm = df_norm.withColumn("name", col("Name")) \
                 .withColumn("domain", 
                    when(col("Website").isNotNull(),
                        when(lower(col("Website")).rlike(r'^https?://'),
                            when(lower(clean_url).rlike(domain_regex), lower(clean_url))
                            .otherwise(lit(None))
                        ).otherwise(col("Website"))
                    ).otherwise(lit(None))
                 )

# Dynamic Prefixing
known_system_keys = ['id', 'systemmodstamp', 'createddate', 'name', 'mockfiletime']
a_cols_to_keep = []
for c in raw_cols:
    if c.lower() not in known_system_keys:
        df_norm = df_norm.withColumn(f"a_{c.lower()}", col(c).cast("string"))
        a_cols_to_keep.append(f"a_{c.lower()}")

final_cols = ['tenant', 'id', 'source_system', 'source_system_object', 'source_key_name', 'source_key_value', 'data_timestamp', 'created_date', 'created_at', 'updated_at', 'status', 'systemmodstamp', 'domain', 'name', 'extraction_timestamp']
final_cols.extend(a_cols_to_keep)
final_cols = list(dict.fromkeys(final_cols)) # Dedup columns
df_norm = df_norm.select(*final_cols)

# B. Intra-Batch Dedup (NO LAST_VALUE - Matches Prod)
window_spec_dedup = Window.partitionBy("id").orderBy(col("extraction_timestamp").desc(), col("data_timestamp").desc())
df_final = df_norm.withColumn("rk", row_number().over(window_spec_dedup)).filter(col("rk") == 1).drop("rk", "extraction_timestamp")

# C. Explicit Schema Evolution
target_schema = spark.table("test_bronze.crm_accounts").schema
target_cols_lower = [f.name.lower() for f in target_schema]

new_cols = []
for f in df_final.schema:
    if f.name.lower() not in target_cols_lower:
        new_cols.append(f"`{f.name}` {f.dataType.simpleString()}")
        
if new_cols:
    print(f"✨ Evolving Schema: Adding {len(new_cols)} new columns to test_bronze.crm_accounts")
    spark.sql(f"ALTER TABLE test_bronze.crm_accounts ADD COLUMNS ({', '.join(new_cols)})")

# D. Dynamic MERGE Upsert
df_final.createOrReplaceTempView("batch_updates")
merge_cols = df_final.columns

update_cols = [c for c in merge_cols if c not in ['tenant', 'id', 'source_system', 'source_system_object', 'source_key_name', 'source_key_value', 'created_at', 'status']]
update_set_parts = []
for c in update_cols:
    if c in ["updated_at", "data_timestamp"]:
        update_set_parts.append(f"target.`{c}` = source.`{c}`")
    else:
        # COALESCE handles the "IGNORE NULLS" requirement natively
        update_set_parts.append(f"target.`{c}` = COALESCE(source.`{c}`, target.`{c}`)")
        
update_set_clause = ", ".join(update_set_parts) + ", target.status = 'updated'"

insert_cols = ", ".join([f"`{c}`" for c in merge_cols])
insert_vals = ", ".join([f"source.`{c}`" if c != 'status' else "'new'" for c in merge_cols])

spark.sql(f"""
    MERGE INTO test_bronze.crm_accounts AS target
    USING batch_updates AS source
    ON target.id = source.id
    WHEN MATCHED AND source.data_timestamp > COALESCE(target.data_timestamp, CAST('1900-01-01' AS TIMESTAMP)) THEN
        UPDATE SET {update_set_clause}
    WHEN NOT MATCHED THEN
        INSERT ({insert_cols})
        VALUES ({insert_vals})
""")

# ---------------------------------------------------------
# 4. Validation & Assertions
# ---------------------------------------------------------
print("\n📊 Validating Results against Test Cases...\n")
results = spark.table("test_bronze.crm_accounts").collect()
res_dict = {row["source_key_value"]: row for row in results}

errors = []

# TC1: Intra-batch dedup (Newest Extraction Time Wins)
if res_dict["DUP456"]["name"] != "Duplicate Co Updated":
    errors.append("❌ TC1 Failed: Intra-batch dedup kept the wrong record name.")

# TC2: Partial Update (MERGE COALESCE) & Status check
if res_dict["EXISTING123"]["name"] != "Old Name":
    errors.append("❌ TC2 Failed: MERGE update failed to COALESCE properly. Name overwritten with NULL.")
if res_dict["EXISTING123"]["a_industry"] != "Tech":
    errors.append(f"❌ TC2 Failed: MERGE failed to apply new industry. Got {res_dict['EXISTING123']['a_industry']}")
if res_dict["EXISTING123"]["status"] != "updated":
    errors.append("❌ TC2 Failed: Status not correctly flagged as 'updated'.")

# TC3: Stale Data Guard (MERGE)
if res_dict["STALE_TEST"]["name"] != "Current Name":
    errors.append("❌ TC3 Failed: Stale record overwrote fresh data.")

# TC4: Brand New Insert & Schema Evolution
if "NEW789" not in res_dict:
    errors.append("❌ TC4 Failed: Brand new record was not inserted.")
elif res_dict["NEW789"]["domain"] != "google.com":
    errors.append(f"❌ TC4 Failed: Domain Regex failed. Got {res_dict['NEW789']['domain']}")
elif "a_risk_level__c" not in res_dict["NEW789"] or res_dict["NEW789"]["a_risk_level__c"] != "High Risk":
    errors.append("❌ TC4 Failed: Schema Evolution failed to add custom column 'a_risk_level__c'.")
elif res_dict["NEW789"]["status"] != "new":
    errors.append("❌ TC4 Failed: Brand new record status is not 'new'.")

# TC5: Fallback Domain logic
if "BAD_URL" in res_dict and res_dict["BAD_URL"]["domain"] != "not-a-valid-url":
    errors.append(f"❌ TC5 Failed: Domain fallback failed on bad URL. Got {res_dict['BAD_URL']['domain']}")

if errors:
    for e in errors:
        print(e)
    raise Exception("Test Suite Failed!")
else:
    print("✅ ALL TEST CASES PASSED!")
    print("✅ 1. Intra-batch deduplication correctly kept the latest record based on extraction time.")
    print("✅ 2. Partial updates successfully ignored NULLs via COALESCE during the MERGE.")
    print("✅ 3. Cross-batch stale data was blocked by the timestamp guard.")
    print("✅ 4. Schema dynamically evolved to capture `a_risk_level__c`.")
    print("✅ 5. Status logic perfectly tagged 'new' and 'updated' rows.")
    print("✅ 6. Domain extraction regex successfully processed clean and dirty URLs.\n")
    
display(spark.table("test_bronze.crm_accounts").select("source_key_value", "name", "a_industry", "domain", "status", "a_risk_level__c").orderBy("source_key_value"))

In [0]:
display(spark.table("bronze.cust_001.crm_accounts").limit(5))

In [0]:
spark.table("bronze.cust_001.crm_accounts").printSchema()

In [0]:
from pyspark.sql.functions import col, to_timestamp

print("🧪 Starting Timestamp Format Verification Test...\n")

# 1. Define the two strings you are comparing
landing_zone_string = "2026-03-16T12:59:12.000Z"
bronze_table_string = "2026-03-16T12:59:12.000+00:00"

# 2. Create a mock dataframe
data = [(landing_zone_string, bronze_table_string)]
df = spark.createDataFrame(data, ["landing_str", "bronze_str"])

# 3. Cast both strings into native Spark Timestamps and compare them
df_test = df.select(
    col("landing_str"),
    col("bronze_str"),
    to_timestamp(col("landing_str")).alias("landing_ts"),
    to_timestamp(col("bronze_str")).alias("bronze_ts")
).withColumn("is_exact_match", col("landing_ts") == col("bronze_ts"))

# Display the visual proof
display(df_test)

# 4. Programmatic Assertion
result = df_test.first()

if result["is_exact_match"]:
    print("✅ TEST PASSED!")
    print("Spark successfully parsed both strings into identical timestamp objects.")
    print(f"Format 1: {landing_zone_string}")
    print(f"Format 2: {bronze_table_string}")
    print("They represent the exact same moment in time.")
else:
    raise Exception("❌ TEST FAILED: The timestamps do not match.")